# RLTF-SD Training Pipeline (Kaggle 2xT4)
---
This notebook runs the full RLTF-SD training loop on Kaggle's 2xT4 GPUs.

**What this does:**
1. Installs dependencies and sets up the Gemini API key.
2. Loads a small math model (Qwen2.5-Math-1.5B-Instruct) that fits in T4 VRAM.
3. Loads the MATH-500 dataset.
4. Runs the custom RLTF-SD trainer from `src/trainer.py`.

**Estimated time:** ~3-6 hours for 1 epoch on MATH-500 (500 prompts × 8 generations each).


In [ ]:
# Install project dependencies (run once)
!pip install -q torch transformers datasets peft google-genai python-dotenv

import sys
sys.path.insert(0, "/kaggle/working/rltf")  # Adjust if your repo is cloned elsewhere


### 1. Set Up API Key

In [ ]:
import os
from google.colab import userdata

# On Kaggle, add your GEMINI_API_KEY as a Secret in the sidebar
# then access it like this:
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("API key set" if os.environ.get("GEMINI_API_KEY") else "WARNING: No API key found!")


### 2. Load Model and Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Qwen2.5-Math-1.5B fits comfortably on a single T4 (16GB).
# With LoRA we only train ~2M parameters out of 1.5B.
# If you have more VRAM, try "Qwen/Qwen2.5-Math-7B-Instruct".
model_id = "Qwen/Qwen2.5-Math-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # T4 supports fp16 natively (not bf16)
    device_map="auto",
)
print(f"Model loaded on: {model.device}")


### 3. Load Dataset

In [ ]:
from src.data import load_data

dataset = load_data()
train_dataset = dataset["test"]  # MATH-500 only has a test split (500 problems)
print(f"Training on {len(train_dataset)} problems")
print(f"Example: {train_dataset[0]['prompt'][:120]}...")


### 4. Initialize Trainer

In [ ]:
from src.trainer import RLTF_SD_Trainer

trainer = RLTF_SD_Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    max_completion_length=512,   # Keep short to save VRAM and time
    group_size=4,                # Paper uses 8, but 4 saves memory on T4
    gamma=1.0,                   # Discount factor for return R = r0 + gamma*r1
    rl_coeff=0.1,                # Paper value: weight of RL loss relative to SD loss
    lr=2e-5,                     # Paper value
    temperature=0.7,
    lora_rank=32,                # Paper value
)


### 5. Train!

In [ ]:
trained_model = trainer.train(
    num_epochs=1,
    log_every=5,
    save_every=100,
    save_dir="/kaggle/working/rltf_checkpoints",
)
print("Training complete!")


### 6. Save Final Model

In [ ]:
trained_model.save_pretrained("/kaggle/working/rltf_final")
tokenizer.save_pretrained("/kaggle/working/rltf_final")
print("Final model saved!")
